# Validación LINE — Solo Cercanías

**Parámetros:** `init_scale=0.5`, `num_negative=1`, `lr=0.05`, `num_epochs_fo=500`, `num_epochs_so=500`, `dim=32`

Pruebas clave:
1. Inspección del grafo
2. Matriz de adyacencia (similitud temporal)
3. Entrenamiento LINE
4. Checks: varianza FO/SO, correlación Spearman tiempo↔distancia
5. **Heatmap de similitud coseno (ordenado por línea)**
6. **Vecinos más cercanos para estaciones clave**
7. PCA interactivo
8. Expansión a punto_acceso y guardado

In [ ]:
import sys, time, pickle
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import spearmanr
from scipy.spatial.distance import cdist
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
import plotly.graph_objects as go
from IPython.display import HTML

sys.path.insert(0, '.')
from station_line import StationLINE

CHECK_DIR = Path('../../cubo_izquierdo/check_transbordo')

with open(CHECK_DIR / 'cercanias_multidigraph_sin_civis.pkl', 'rb') as f:
    G = pickle.load(f)

print(f'Cercanías: {G.number_of_nodes()} nodos, {G.number_of_edges()} aristas')
print('Módulo importado correctamente')

---
## 1 — Inspección del grafo

In [ ]:
nodo  = list(G.nodes(data=True))[0]
arista = list(G.edges(data=True))[0]
print(f'Ejemplo nodo:   {nodo[0]}')
print(f'  Atributos:    {nodo[1]}')
print(f'Ejemplo arista: {arista[0]} → {arista[1]}')
print(f'  Atributos:    {arista[2]}')

# Distribución de tiempos de viaje
pesos = [d.get('weight', 0) for _, _, d in G.edges(data=True)]
fig, ax = plt.subplots(figsize=(9, 3))
ax.hist(pesos, bins=25, color='steelblue', alpha=0.8, edgecolor='white')
ax.set_xlabel('Tiempo de viaje (min)')
ax.set_ylabel('Nº aristas')
ax.set_title(f'Cercanías — distribución de tiempos entre estaciones\n'
             f'(media={np.mean(pesos):.1f} min, máx={np.max(pesos):.1f} min, {len(pesos)} aristas)')
plt.tight_layout()
plt.show()

---
## 2 — Matriz de adyacencia

In [ ]:
m_tmp = StationLINE(dim=4, num_epochs_fo=1, num_epochs_so=1)
nodes, adj = m_tmp._build_adjacency(G)
n = len(nodes)
n_con = int((np.triu(adj, k=1) > 0).sum())
sim_vals = adj[adj > 0]

print(f'Shape:              {adj.shape}')
print(f'Simétrica:          {np.allclose(adj, adj.T)}')
print(f'Pares conectados:   {n_con} / {n*(n-1)//2} ({n_con/(n*(n-1)//2)*100:.1f}%)')
print(f'Similitud mín:      {sim_vals.min():.4f}  (= 1 / {1/sim_vals.min():.1f} min)')
print(f'Similitud máx:      {sim_vals.max():.4f}  (= 1 / {1/sim_vals.max():.1f} min)')
print(f'Similitud media:    {sim_vals.mean():.4f}')

---
## 3 — Entrenamiento LINE

In [ ]:
model = StationLINE(
    dim=32,
    num_negative=1,
    lr=0.05,
    num_epochs_fo=500,
    num_epochs_so=500,
    init_scale=0.5,
    random_state=42,
)

t0  = time.time()
emb = model.fit_transform(G)
print(f'Completado: {emb.shape}  ({time.time()-t0:.1f}s)')
emb.head(3).round(4)

---
## 4 — Checks de validación

In [ ]:
fo_cols = [c for c in emb.columns if c.startswith('fo_')]
so_cols = [c for c in emb.columns if c.startswith('so_')]

ok_shape = len(emb) == G.number_of_nodes() and emb.shape[1] == 2 * model.dim
ok_nan   = not emb.isnull().any().any() and np.isfinite(emb.values).all()
var_fo   = emb[fo_cols].values.var()
var_so   = emb[so_cols].values.var()

print(f'Shape ({G.number_of_nodes()} × {2*model.dim}): {"✓" if ok_shape else "✗"}')
print(f'Sin NaN/Inf:                  {"✓" if ok_nan else "✗"}')
print(f'Varianza FO: {var_fo:.5f}  SO: {var_so:.5f}')
print()

In [ ]:
# Correlación de Spearman: tiempo de viaje ↔ distancia coseno en embedding
# Si LINE funciona bien: pares con más tiempo de viaje → embeddings más alejados (r > 0)

with np.errstate(divide='ignore', invalid='ignore'):
    T = np.where(model.adj_ > 0, 1.0 / model.adj_, 0.0)

X = emb[[c for c in emb.columns if c.startswith('fo_') or c.startswith('so_')]].values.astype(float)
D = cdist(X, X, metric='cosine')

idx  = np.triu_indices(len(model.station_index_), k=1)
mask = T[idx] > 0
r, p = spearmanr(T[idx][mask], D[idx][mask])

print(f'Spearman tiempo↔distancia coseno: r={r:.3f}  (p={p:.2e})  sobre {mask.sum()} pares')
if r > 0:
    print('✓ Correlación positiva: más tiempo → más alejados en embedding (esperado)')
else:
    print('✗ Correlación negativa — revisar parámetros')

In [ ]:
# Similitud intra-línea vs inter-línea
# Las estaciones de la misma línea deben ser más similares entre sí

def get_line(node, G):
    aps = G.nodes[node].get('access_point_codes', set())
    if isinstance(aps, str):
        aps = {aps}
    for ap in aps:
        parts = ap.split('_')
        if len(parts) >= 2:
            return parts[1]
    return 'L?'

X_full = emb.values.astype(float)
sim    = cosine_similarity(X_full)
lines  = np.array([get_line(s, G) for s in emb.index])
n_st   = len(lines)

intra, inter = [], []
for i in range(n_st):
    for j in range(i+1, n_st):
        (intra if lines[i] == lines[j] else inter).append(sim[i, j])

print(f'Similitud coseno media intra-línea: {np.mean(intra):.4f}')
print(f'Similitud coseno media inter-línea: {np.mean(inter):.4f}')
print(f'intra > inter → {"✓" if np.mean(intra) > np.mean(inter) else "✗"}')

---
## 5 — Heatmap similitud coseno (ordenado por línea)

In [ ]:
# Ordenar por línea para ver estructura de bloques diagonales
stations  = list(emb.index)
lines_all = [get_line(s, G) for s in stations]
order     = np.argsort(lines_all)

X_sorted  = X_full[order]
sim_sorted = cosine_similarity(X_sorted)
labels_x  = [stations[i] for i in order]
labels_y  = [f'{stations[i]}  ({lines_all[i]})' for i in order]

n_st = len(stations)
fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(sim_sorted, cmap='RdBu', vmin=-1, vmax=1, aspect='auto')
ax.set_title(
    f'Similitud coseno entre estaciones cercanías\n'
    f'(ordenadas por línea · init_scale=0.5 · FO+SO 500 épocas)',
    fontsize=10
)
ax.set_xticks(range(n_st))
ax.set_xticklabels(labels_x, rotation=90, fontsize=4.5)
ax.set_yticks(range(n_st))
ax.set_yticklabels(labels_y, fontsize=4.5)
plt.colorbar(im, ax=ax, label='Similitud coseno', shrink=0.5)

# Separadores de línea
prev_line = lines_all[order[0]]
for k, i in enumerate(order):
    if lines_all[i] != prev_line:
        ax.axhline(k - 0.5, color='black', linewidth=0.6, alpha=0.6)
        ax.axvline(k - 0.5, color='black', linewidth=0.6, alpha=0.6)
        prev_line = lines_all[i]

plt.tight_layout()
plt.show()

In [ ]:
# Versión interactiva (hover para identificar estaciones)
fig_p = go.Figure(go.Heatmap(
    z=sim_sorted,
    x=labels_x,
    y=labels_y,
    colorscale='RdBu',
    zmid=0,
    hovertemplate='%{y}<br>%{x}<br>sim=%{z:.3f}<extra></extra>',
))
fig_p.update_layout(
    title='Cercanías — Similitud coseno embeddings LINE (ordenado por línea)',
    height=700,
    xaxis=dict(showticklabels=False),
    yaxis=dict(showticklabels=False),
)
HTML(fig_p.to_html(include_plotlyjs='cdn', full_html=False))

---
## 6 — Vecinos más cercanos para estaciones clave

In [ ]:
# Clave: si LINE funciona bien, los vecinos deben ser
# estaciones TOPOLÓGICAMENTE cercanas (misma línea, tramo adyacente)
# — no necesariamente en la misma zona geográfica.

KEY_HUBS = [
    'SOL',
    'MADRID ATOCHA CERCANÍAS',
    'MADRID CHAMARTÍN-CLARA CAMPOAMOR',
    'PRÍNCIPE PÍO',
    'NUEVOS MINISTERIOS',
    'CANTOBLANCO UNIVERSIDAD',
    'ALCALÁ DE HENARES',
    'PARLA',
    'GETAFE CENTRO',
    'TRES CANTOS',
]
K = 8

station_list = list(emb.index)
sim_all = cosine_similarity(X_full)

print(f'Top-{K} vecinos más cercanos en embedding LINE (cercanías)\n')

for hub in KEY_HUBS:
    matches = [s for s in station_list if hub.upper() in s.upper()]
    if not matches:
        print(f'  {hub}: no encontrada\n')
        continue
    station = matches[0]
    idx     = station_list.index(station)
    sims    = sim_all[idx]
    top_idx = np.argsort(sims)[::-1][1:K+1]

    print(f'{station}:')
    for rank, j in enumerate(top_idx, 1):
        line_j = get_line(station_list[j], G)
        print(f'  {rank}. {station_list[j]:<45} ({line_j})  sim={sims[j]:.4f}')
    print()

In [ ]:
# Visualización: perfil FO+SO de la estación + sus vecinos
# Si los primeros 32 dims (FO) capturan estructura local y
# los últimos 32 (SO) capturan estructura global, los vecinos
# deben parecerse en ambas mitades.

VIS_HUBS = ['SOL', 'MADRID ATOCHA CERCANÍAS', 'CANTOBLANCO UNIVERSIDAD', 'PARLA']
K_VIS = 3

for hub in VIS_HUBS:
    matches = [s for s in station_list if hub.upper() in s.upper()]
    if not matches:
        continue
    station = matches[0]
    idx     = station_list.index(station)
    sims    = sim_all[idx]
    top_idx = np.argsort(sims)[::-1][1:K_VIS+1]
    group   = [station] + [station_list[j] for j in top_idx]

    fig, axes = plt.subplots(1, len(group), figsize=(len(group) * 4.5, 3.5), sharey=True)

    for k, (ax, s) in enumerate(zip(axes, group)):
        vec    = emb.loc[s].values
        dims   = range(len(vec))
        fo_v   = [vec[d] if d < 32 else 0 for d in dims]
        so_v   = [vec[d] if d >= 32 else 0 for d in dims]
        ax.bar(dims, fo_v, color='steelblue', alpha=0.8, label='FO' if k == 0 else '')
        ax.bar(dims, so_v, color='darkorange', alpha=0.8, label='SO' if k == 0 else '')
        ax.axvline(31.5, color='black', linewidth=0.8, linestyle='--')
        sim_v = sims[station_list.index(s)] if k > 0 else 1.0
        ttl   = f'{s[:25]}'
        if k > 0:
            ttl += f'\n({get_line(s, G)})  sim={sim_v:.3f}'
        ax.set_title(ttl, fontsize=7.5)
        ax.set_xlabel('Dimensión', fontsize=7)
        if k == 0:
            ax.legend(fontsize=7)

    fig.suptitle(f'{station} — embedding FO (azul) + SO (naranja) vs vecinos', fontsize=9)
    plt.tight_layout()
    plt.show()

---
## 7 — PCA interactivo (coloreado por línea)

In [ ]:
pca    = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(X_full)
unique_lines = sorted(set(lines_all))

fig = go.Figure()
for line in unique_lines:
    mask = np.array(lines_all) == line
    fig.add_trace(go.Scatter(
        x=coords[mask, 0],
        y=coords[mask, 1],
        mode='markers+text',
        name=line,
        text=[s for s, m in zip(station_list, mask) if m],
        textposition='top center',
        textfont=dict(size=7),
        hovertemplate='<b>%{text}</b><br>PC1: %{x:.3f}<br>PC2: %{y:.3f}<extra></extra>',
        marker=dict(size=9, opacity=0.85),
    ))

fig.update_layout(
    title=f'Cercanías — PCA embedding LINE (FO + SO)<br>'
          f'<sup>PC1={pca.explained_variance_ratio_[0]*100:.1f}%  '
          f'PC2={pca.explained_variance_ratio_[1]*100:.1f}%  |  '
          f'init_scale=0.5 · lr=0.05 · FO+SO 500 épocas · dim=32</sup>',
    xaxis_title='PC1',
    yaxis_title='PC2',
    legend_title='Línea',
    height=650,
    hovermode='closest',
)
HTML(fig.to_html(include_plotlyjs='cdn', full_html=False))

In [ ]:
# PCA separado FO vs SO para ver cuál captura más estructura
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colores   = plt.cm.tab10(np.linspace(0, 1, len(unique_lines)))
color_map = {l: c for l, c in zip(unique_lines, colores)}

for ax, part, col_prefix in zip(axes, ['First-order (FO)', 'Second-order (SO)'], ['fo_', 'so_']):
    X_part   = emb[[c for c in emb.columns if c.startswith(col_prefix)]].values.astype(float)
    pca_part = PCA(n_components=2, random_state=42)
    coords_p = pca_part.fit_transform(X_part)
    for s, line, (x, y) in zip(station_list, lines_all, coords_p):
        ax.scatter(x, y, color=color_map[line], s=40, alpha=0.8)
        ax.annotate(s[:12], (x, y), fontsize=4, alpha=0.7)
    ax.set_title(f'{part}\nPC1={pca_part.explained_variance_ratio_[0]*100:.1f}%  '
                 f'PC2={pca_part.explained_variance_ratio_[1]*100:.1f}%', fontsize=9)
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')

# Leyenda común
from matplotlib.patches import Patch
patches = [Patch(color=color_map[l], label=l) for l in unique_lines]
fig.legend(handles=patches, loc='lower center', ncol=len(unique_lines), fontsize=7,
           bbox_to_anchor=(0.5, -0.02))
fig.suptitle('PCA FO vs SO — Cercanías LINE', fontsize=10)
plt.tight_layout()
plt.show()

---
## 8 — Expansión a punto_acceso y guardado

In [ ]:
ap = model.to_punto_acceso(G)
print(f'Estaciones: {len(emb):>3}  →  puntos de acceso: {len(ap):>3}')

# Verificar prefijo 04_
ok = ap.index.str.startswith('04_').all()
print(f'Todos con prefijo 04_: {"✓" if ok else "✗"}')

# Verificar herencia (cada AP hereda el vector exacto de su estación)
emb_cols    = [c for c in emb.columns if c.startswith('fo_') or c.startswith('so_')]
mismatches  = 0
for node, data in G.nodes(data=True):
    aps = data.get('access_point_codes', set())
    if isinstance(aps, str):
        aps = {aps}
    if node not in emb.index:
        continue
    vec = emb.loc[node, emb_cols].values.astype(float)
    for ap_code in aps:
        if ap_code in ap.index and not np.allclose(ap.loc[ap_code, emb_cols].values.astype(float), vec):
            mismatches += 1

print(f'Herencia estación→AP: {"✓" if mismatches == 0 else f"✗ ({mismatches} desajustes)"}')
ap.head()

In [ ]:
out_dir = Path('output')
out_dir.mkdir(exist_ok=True)

emb.to_parquet(out_dir / 'line_cercanias.parquet')
emb.round(6).to_csv(out_dir / 'line_cercanias.csv')
ap.to_parquet(out_dir / 'line_cercanias_ap.parquet')

print(f'Guardado en: {out_dir.resolve()}')
print(f'  line_cercanias.parquet     {emb.shape}')
print(f'  line_cercanias_ap.parquet  {ap.shape}')